# 10月回测结果与数据质量复核
研究参考结果；严格分钟数据验收未通过。完整复跑入口为同目录 run_sparse_reference.py。
本复核本通过标准Python逐单元执行并保存输出；当前环境没有Jupyter内核验证工具。

In [1]:
from pathlib import Path
import json, hashlib
ROOT = Path('/Users/wangxiangyu/Documents/myquan')
OUT = Path('/Users/wangxiangyu/Documents/myquan/backtest/strategy16_core_leader/review_2026-10-01_2026-10-16_intraday_150100')
r = json.loads((OUT / 'research_result.json').read_text())
for source in r['sources'] + r['code_sources']:
    assert hashlib.sha256((ROOT / source['path']).read_bytes()).hexdigest() == source['sha256'], source['path']
print('输入及代码文件哈希检查通过；', len(r['sessions']), '个交易日；', r['minute_rows'], '根分钟记录')

输入及代码文件哈希检查通过； 7 个交易日； 56607 根分钟记录


In [2]:
initial = r['configuration']['initial_cash']
previous = initial
compound = 1.0
for day in r['daily_returns']:
    assert abs(day['equity'] - previous - day['day_pnl']) < 0.011
    assert abs((day['equity'] / previous - 1) * 100 - day['day_return_pct']) < 1e-9
    compound *= 1 + day['day_return_pct'] / 100
    previous = day['equity']
    print(day['date'], f"{day['day_pnl']:+.2f}元 {day['day_return_pct']:+.4f}%")
assert abs(sum(d['day_pnl'] for d in r['daily_returns']) - r['accounting']['net_profit']) < 0.02
assert abs(compound - previous / initial) < 1e-9
assert abs(r['accounting']['cash'] + r['accounting']['market_value'] - previous) < 0.02
print('每日收益、复利与资金市值核对通过')

2026-10-08 -4312.50元 -1.4375%
2026-10-09 -1777.94元 -0.6013%
2026-10-12 -2464.27元 -0.8384%
2026-10-13 -2397.64元 -0.8227%
2026-10-14 -6367.13元 -2.2028%
2026-10-15 -2681.12元 -0.9485%
2026-10-16 +1503.07元 +0.5368%
每日收益、复利与资金市值核对通过


In [3]:
print('数据状态:', r['status'])
print('相对正常供应商分钟标签缺失:', r['missing_source_labels'])
print('最少历史日线预热行数:', r['minimum_daily_warmup_rows'])
for caveat in r['caveats']:
    print('-', caveat)

数据状态: research_only_sparse_minutes_strict_acceptance_failed
相对正常供应商分钟标签缺失: 37
最少历史日线预热行数: 52
- 研究参考：原严格入口因三联锻造10月15日仅229根分钟记录而终止；本结果保留全部34只，未补造分钟。
- 7个交易日均有数据，但逐分钟覆盖不完整；稀疏记录原因未确认，不能声称完整精确回测。
- 使用当前冻结股票池、个股阈值；thr_2026拟合截至2026-10-13，与评价窗重叠，非样本外。
- 不包含因子29竞价买入；缺历史竞价tick与完整报价事件，不能精确复现当前实时策略。
- 30万元空仓开始，未继承9月持仓，不等于实际账户收益。
- 沿用现有分钟回测内核、T+1、费用和滑点模型，同分钟竞争按固定池顺序；未证明与报价交易逐笔一致。
- 10月尚未结束，本次只到10月16日15:00；未计算未来交易日。
